<a href="https://colab.research.google.com/github/LegalIntermediaSL/Nautica/blob/main/simulaciones/17_generador_test_aleatorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulación 17: Generador de Examen Tipo Test Aleatorio (PER)

El examen oficial del **PER** consta de 45 preguntas tipo test y se admite un máximo de 13 fallos totales, pero además hay que respetar los **criterios eliminatorios** en tres bloques concretos (ver [`titulaciones/PER/INDEX.md`](../titulaciones/PER/INDEX.md)):

*   **Balizamiento (IALA):** 5 preguntas del examen real. Eliminatorio con **más de 2 fallos**.
*   **RIPA:** 10 preguntas del examen real. Eliminatorio con **más de 5 fallos**.
*   **Carta de navegación:** 4 preguntas prácticas. Eliminatorio con más de 2 fallos (no se simula aquí).

Es decir: aunque saques buena nota global, si fallas 3 o más preguntas de Balizamiento, o 6 o más de RIPA, **suspendes el examen igualmente**.

Esta simulación define un pequeño **banco de preguntas** (Balizamiento, RIPA y Nomenclatura náutica) y una función que usa `random.sample()` para escoger N preguntas sin repetición, presentarlas, corregirlas y evaluar si, con ese resultado, superarías los criterios eliminatorios de Balizamiento y RIPA.

In [ ]:
import random

# --- BANCO DE PREGUNTAS (tema, enunciado, opciones, índice de la correcta) ---
banco_preguntas = [
    {"tema": "Balizamiento", "pregunta": "Entrando a puerto (Región A - IALA), ¿de qué color es la marca lateral de ESTRIBOR?",
     "opciones": ["Rojo", "Verde", "Amarillo", "Negro"], "correcta": 1},
    {"tema": "Balizamiento", "pregunta": "Entrando a puerto (Región A - IALA), ¿qué forma tiene la marca lateral de BABOR?",
     "opciones": ["Cónica", "Esférica", "Cilíndrica", "En forma de X"], "correcta": 2},
    {"tema": "Balizamiento", "pregunta": "Una marca cardinal NORTE lleva como marca de tope dos conos negros...",
     "opciones": ["Con los vértices hacia arriba", "Con los vértices hacia abajo",
                  "Opuestos por sus bases (rombo)", "Opuestos por sus vértices (reloj de arena)"], "correcta": 0},
    {"tema": "Balizamiento", "pregunta": "¿Qué ritmo de luz blanca caracteriza a una marca cardinal SUR?",
     "opciones": ["Centelleo rápido continuo (Q)", "Grupo de 3 centelleos rápidos Q(3)",
                  "Grupo de 6 centelleos + 1 destello largo VQ(6)+LFl", "Grupo de 9 centelleos rápidos Q(9)"], "correcta": 2},
    {"tema": "Balizamiento", "pregunta": "La marca de PELIGRO AISLADO se identifica por su color y su marca de tope:",
     "opciones": ["Amarillo con aspa (X) negra", "Negro con bandas rojas y dos esferas negras superpuestas",
                  "Rayas verticales rojas y blancas con una esfera roja", "Amarillo y negro con dos conos negros"], "correcta": 1},
    {"tema": "Balizamiento", "pregunta": "La marca de AGUAS NAVEGABLES (eje de canal, recalada) tiene como marca de tope:",
     "opciones": ["Una esfera roja", "Dos conos negros", "Un cilindro rojo", "Un aspa amarilla"], "correcta": 0},
    {"tema": "Balizamiento", "pregunta": "Las marcas especiales (zonas reguladas, tuberías, regatas) son de color:",
     "opciones": ["Negro y amarillo", "Rojo y blanco", "Amarillo", "Azul y amarillo"], "correcta": 2},
    {"tema": "RIPA", "pregunta": "Según la Regla 13, se considera que un buque \"alcanza\" a otro cuando se le aproxima desde una dirección que forma con su proa un ángulo superior a:",
     "opciones": ["45º a popa del través", "22,5º a popa del través", "90º (justo por el través)", "10º a popa del través"], "correcta": 1},
    {"tema": "RIPA", "pregunta": "En un encuentro de vueltas encontradas (Regla 14) entre dos buques de motor, ambos deben caer a:",
     "opciones": ["Babor", "Estribor", "Cualquiera, según acuerden por VHF", "Ninguno, deben parar máquinas"], "correcta": 1},
    {"tema": "RIPA", "pregunta": "En una situación de cruce (Regla 15) entre dos buques de motor, ¿quién debe ceder el paso?",
     "opciones": ["El que ve al otro por su babor", "El que ve al otro por su estribor",
                  "El buque de mayor eslora", "El que navega más rápido"], "correcta": 1},
    {"tema": "RIPA", "pregunta": "Según la jerarquía de prioridad de paso de la Regla 18, ¿qué buque tiene mayor prioridad?",
     "opciones": ["Buque a vela", "Buque dedicado a la pesca profesional", "Buque sin gobierno", "Buque de propulsión mecánica"], "correcta": 2},
    {"tema": "RIPA", "pregunta": "Un velero que navega con el motor auxiliar encendido (aunque lleve las velas izadas) se considera legalmente:",
     "opciones": ["Buque a vela", "Buque de propulsión mecánica", "Buque sin gobierno", "Buque restringido"], "correcta": 1},
    {"tema": "RIPA", "pregunta": "Una señal acústica de UNA pitada corta significa:",
     "opciones": ["Caigo a estribor", "Caigo a babor", "Doy atrás", "Peligro o duda"], "correcta": 0},
    {"tema": "RIPA", "pregunta": "De noche, un buque SIN GOBIERNO muestra:",
     "opciones": ["Una luz blanca todo el horizonte", "Dos luces rojas en vertical",
                  "Roja-Blanca-Roja en vertical", "Verde sobre blanca"], "correcta": 1},
    {"tema": "RIPA", "pregunta": "Entre dos veleros con el viento por bandas contrarias, debe ceder el paso el que:",
     "opciones": ["Amura a estribor", "Amura a babor", "Va a mayor velocidad", "Está a sotavento"], "correcta": 1},
    {"tema": "Nomenclatura", "pregunta": "La Eslora de Flotación (LWL) es especialmente relevante porque determina, junto al Número de Froude:",
     "opciones": ["La manga máxima permitida", "La velocidad máxima teórica de un casco de desplazamiento",
                  "El calado mínimo de seguridad", "El francobordo reglamentario"], "correcta": 1},
    {"tema": "Nomenclatura", "pregunta": "Si el calado de popa (Ta) es mayor que el calado de proa (Tf), el barco tiene asiento:",
     "opciones": ["Apopante", "Aproante", "Nulo", "Transversal"], "correcta": 0},
    {"tema": "Nomenclatura", "pregunta": "Cuando la proa y la popa están sobre crestas de ola y el centro del barco en un seno, el casco sufre:",
     "opciones": ["Quebranto (hogging)", "Arrufo (sagging)", "Abatimiento", "Escora"], "correcta": 1},
    {"tema": "Nomenclatura", "pregunta": "La zona del costado del barco comprendida entre el través y la popa se llama:",
     "opciones": ["Amura", "Través", "Aleta", "Codaste"], "correcta": 2},
    {"tema": "Nomenclatura", "pregunta": "La distancia vertical desde la línea de flotación hasta la cubierta principal se denomina:",
     "opciones": ["Puntal", "Calado", "Francobordo", "Manga"], "correcta": 2},
]

print(f"Banco de preguntas cargado: {len(banco_preguntas)} preguntas.")
for tema in ("Balizamiento", "RIPA", "Nomenclatura"):
    n = sum(1 for p in banco_preguntas if p["tema"] == tema)
    print(f"  - {tema}: {n} preguntas")

In [ ]:
# Criterios eliminatorios oficiales del PER (sobre el examen real completo)
LIMITES_ELIMINATORIOS = {"Balizamiento": 2, "RIPA": 5}


def generar_examen(banco, n=10, semilla=None):
    """Selecciona N preguntas al azar y SIN repetición usando random.sample."""
    if semilla is not None:
        random.seed(semilla)
    return random.sample(banco, n)


def realizar_examen_interactivo(banco, n=10):
    """Presenta el examen pregunta a pregunta y pide la respuesta por teclado
    (usa esta función ejecutando la celda en Jupyter/VS Code)."""
    preguntas = generar_examen(banco, n)
    aciertos = 0
    fallos_por_tema = {}
    for i, p in enumerate(preguntas, start=1):
        print(f"\nPregunta {i}/{n} [{p['tema']}]: {p['pregunta']}")
        for j, opcion in enumerate(p["opciones"]):
            print(f"   {chr(97 + j)}) {opcion}")
        respuesta = input("Tu respuesta (a/b/c/d): ").strip().lower()
        indice = ord(respuesta[0]) - 97 if respuesta and respuesta[0] in "abcd" else -1
        if indice == p["correcta"]:
            aciertos += 1
            print("Correcto.")
        else:
            print(f"Incorrecto. Respuesta correcta: {p['opciones'][p['correcta']]}")
            fallos_por_tema[p["tema"]] = fallos_por_tema.get(p["tema"], 0) + 1
    return aciertos, fallos_por_tema, preguntas


def evaluar_resultado(aciertos, fallos_por_tema, n):
    nota = aciertos / n * 10
    print(f"\n=== RESULTADO: {aciertos}/{n} aciertos (nota {nota:.1f}/10) ===")
    superado_eliminatorios = True
    for tema, limite in LIMITES_ELIMINATORIOS.items():
        fallos = fallos_por_tema.get(tema, 0)
        estado = "OK" if fallos <= limite else "ELIMINADO"
        if fallos > limite:
            superado_eliminatorios = False
        print(f"  {tema}: {fallos} fallos (máximo permitido: {limite}) -> {estado}")
    print("=> APTO en los criterios eliminatorios simulados." if superado_eliminatorios
          else "=> NO APTO: se ha superado un criterio eliminatorio.")
    return superado_eliminatorios

In [ ]:
# --- DEMOSTRACIÓN AUTOMÁTICA (sin input, para poder ejecutar todo el notebook de un tirón) ---
# En lugar de pedir la respuesta por teclado, simulamos un alumno que responde
# al azar, solo para enseñar cómo funciona la corrección y el criterio eliminatorio.

random.seed(42)  # reproducible
N_PREGUNTAS = 12
examen_demo = generar_examen(banco_preguntas, n=N_PREGUNTAS)

aciertos = 0
fallos_por_tema = {}
for i, p in enumerate(examen_demo, start=1):
    respuesta_simulada = random.randrange(len(p["opciones"]))  # alumno "aleatorio"
    if respuesta_simulada == p["correcta"]:
        aciertos += 1
    else:
        fallos_por_tema[p["tema"]] = fallos_por_tema.get(p["tema"], 0) + 1
    print(f"{i:2d}. [{p['tema']:<12}] {p['pregunta'][:60]}... "
          f"-> {'OK' if respuesta_simulada == p['correcta'] else 'FALLO'}")

evaluar_resultado(aciertos, fallos_por_tema, N_PREGUNTAS)

# Para hacer el examen tú mismo/a de verdad, ejecuta en otra celda:
#   aciertos, fallos_por_tema, preguntas = realizar_examen_interactivo(banco_preguntas, n=15)
#   evaluar_resultado(aciertos, fallos_por_tema, 15)

## Conclusión

Superar el PER no consiste solo en acumular aciertos: **Balizamiento y RIPA son criterios eliminatorios independientes** de la nota global, tal como recoge [`titulaciones/PER/INDEX.md`](../titulaciones/PER/INDEX.md). Puedes tener un examen brillante en el resto de temas y suspender igualmente por fallar 3 preguntas de Balizamiento o 6 de RIPA. Este generador es solo una demostración didáctica con un banco reducido de preguntas; para un simulacro realista consulta [`titulaciones/PER/simulacro_examen.md`](../titulaciones/PER/simulacro_examen.md) y las 45 preguntas oficiales del examen.